# Part 3 — Exploratory analysis

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path("../src").resolve()))


import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path("../data")
PROCESSED_DIR = DATA_DIR / "processed"
FIGURES_DIR = Path("../figures")

from summaries import summarize_group, summarize_numeric

In [2]:
## Loading Clean Data For Analysis
Express_df = pd.read_csv(PROCESSED_DIR / "ExpressMart_processed.csv", parse_dates=["order_datetime"])


In [3]:
Express_df.shape

(185652, 12)

In [4]:
Express_df.columns

Index(['order_id', 'product', 'quantity_ordered', 'price_each',
       'purchase_address', 'sales', 'order_datetime', 'month', 'hour', 'city',
       'state', 'city_state'],
      dtype='str')

In [5]:
Express_df.dtypes

order_id                     int64
product                        str
quantity_ordered             int64
price_each                 float64
purchase_address               str
sales                      float64
order_datetime      datetime64[us]
month                        int64
hour                         int64
city                           str
state                          str
city_state                     str
dtype: object

Data Questions
1. Which location brings the highest Sales ?
2. At what hour of day do sales go up? / Identity the peak sales period
3. What does a typical sale look like, and does price affect sales?
4. Which products do customers by most and which products bring in higest sales


1. Which location brings the highest Sales ?

In [6]:
summarize_group(Express_df, "city_state")

,city_state,purchases,quantity_sold,total_sales,avg_sale,pct_of_total_sales
0,"San Francisco, CA",44650,50156,8252258.67,184.821023,0.239
1,"Los Angeles, CA",29563,33246,5447304.29,184.260876,0.158
2,"New York City, NY",24837,27887,4660526.52,187.644503,0.135
3,"Boston, MA",19897,22490,3657300.76,183.811668,0.106
4,"Atlanta, GA",14863,16584,2794199.07,187.996977,0.081
5,"Dallas, TX",14795,16705,2763659.01,186.796824,0.080
6,"Seattle, WA",14712,16533,2744896.03,186.575315,0.080
7,"Portland, OR",9996,11289,1869857.57,187.060581,0.054
8,"Austin, TX",9888,11135,1817544.35,183.813142,0.053
9,"Portland, ME",2451,2746,449321.38,183.321656,0.013


San Francisco is the top location at about $8.25 million, roughly 24% of total revenue ($34.5 million). Los Angeles ($5.45M) and New York City ($4.66M) follow. Portland, ME is the lowest at about $0.45 million, around 1.3% of the total, so the top location earns roughly 18 times as much as the bottom one.

The mean value per row is almost the same in every city (about $183 to $188). So the differences in revenue come from how many items each city sells, not from customers spending more per item. The ranking by quantity sold is almost the same as by revenue. Atlanta and Dallas swap places, which means Atlanta sold slightly fewer items but earned more. 

2. At what hour of day do sales go up? / Identity the peak sales period

In [7]:
by_hour = summarize_group(Express_df, "hour").sort_values("hour").reset_index(drop = True)
by_hour[["hour", "quantity_sold", "total_sales"]]

,hour,quantity_sold,total_sales
0,0,4404,710887.98
1,1,2607,458258.99
2,2,1393,232574.51
3,3,921,144683.04
4,4,931,162281.15
5,5,1490,230549.93
6,6,2809,447963.01
7,7,4547,744067.32
8,8,6998,1192309.20
9,9,9808,1638784.04


In [8]:
by_hour.sort_values("quantity_sold", ascending=False).reset_index(drop=True)[["hour", "quantity_sold", "total_sales"]]

,hour,quantity_sold,total_sales
0,19,14451,2411971.14
1,12,14188,2314359.85
2,11,13986,2296619.84
3,18,13785,2218374.01
4,20,13758,2280784.36
5,13,13671,2152369.98
6,14,12343,2082513.70
7,10,12292,1942988.08
8,21,12228,2040790.48
9,17,12214,2126553.97


Orders show two clear sales peaks throughout the day. The first occurs at noon, with 14,188 units sold and total sales of $2,314,359.85. The second and highest peak occurs at 7 PM, with 14,451 units sold and total sales of $2,411,971.14. Between these peaks, sales experience an afternoon dip, reaching their lowest point at 3 PM, approximately 19% below the noon peak. The quietest time of the day is 3 AM, when only 921 units are sold.

These patterns may be explained by customers' daily routines, such as lunchtime shopping and increased activity in the early evening. Based on these findings, the store could schedule additional staff during the peak hours around noon and early evening to serve customers more efficiently. It could also introduce targeted promotions during these periods to maximize sales, while using the quieter hours to manage staffing and other operational activities more effectively.

3. What does a typical sale look like, and does price affect sales?

In [9]:
summarize_numeric(Express_df, "sales")

mean       185.60
median      14.95
min          2.99
max       3400.00
Name: sales, dtype: float64

In [10]:
summarize_numeric(Express_df, "price_each")

mean       184.51
median      14.95
min          2.99
max       1700.00
Name: price_each, dtype: float64

In [11]:
Express_df["quantity_ordered"].value_counts().sort_index()

quantity_ordered
1    168261
2     13319
3      2919
4       805
5       236
6        80
7        24
8         5
9         3
Name: count, dtype: int64

In [12]:
(Express_df["quantity_ordered"].value_counts(normalize=True).sort_index()).round(1)

quantity_ordered
1    0.9
2    0.1
3    0.0
4    0.0
5    0.0
6    0.0
7    0.0
8    0.0
9    0.0
Name: proportion, dtype: float64

In [13]:
by_product = Express_df.groupby("product").agg(
    price=("price_each", "mean"),
    quantity_sold=("quantity_ordered", "sum"),
)
by_product.sort_values("price")

,price,quantity_sold
product,,
AAA Batteries (4-pack),2.99,30981
AA Batteries (4-pack),3.84,27615
USB-C Charging Cable,11.95,23927
Wired Headphones,11.99,20520
Lightning Charging Cable,14.95,23163
Bose SoundSport Headphones,99.99,13427
20in Monitor,109.99,4123
27in FHD Monitor,149.99,7538
Apple Airpods Headphones,150.00,15633


In [14]:
by_product[["price", "quantity_sold"]].corr()

,price,quantity_sold
price,1.000000,-0.601277
quantity_sold,-0.601277,1.000000


Sales values are strongly skewed to the right. The mean is $185.60 but the median is only $14.95, and the largest is $3,400. A few expensive items (laptops, phones) pull the mean far above what most purchases look like.

About 91% of purchases have a quantity of 1 (168,261 rows), and counts drop quickly for larger quantities. So large sales values come from the product's price, not from buying many copies.

Across the 19 products, `price_each` and `quantity_sold` have a correlation of about -0.60: more expensive products tend to have a lower quantity sold. This is a moderate pattern from only 19 products, and some break it (Apple AirPods at $150 have a quantity sold of 15,633). Correlation also does not show that price causes the difference.

The products with the highest `quantity_sold` are cheap: AAA Batteries (30,981), AA Batteries (27,615) and USB-C Charging Cable (23,927). The highest `total_sales` come from different products: the Macbook Pro Laptop ($8.03M), iPhone ($4.79M) and ThinkPad Laptop ($4.13M). The products that sell the most items and the products that earn the most money are not the same.

4. Which products do customers by most and which products bring in higest sales

In [15]:
product_summary = summarize_group(Express_df, "product")

In [16]:
top_quantity = (
    product_summary
    .sort_values("quantity_sold", ascending=False)
    .head(5)
    .reset_index(drop=True)
)
top_quantity[["product", "quantity_sold", "total_sales"]]

,product,quantity_sold,total_sales
0,AAA Batteries (4-pack),30981,92633.19
1,AA Batteries (4-pack),27615,106041.60
2,USB-C Charging Cable,23927,285927.65
3,Lightning Charging Cable,23163,346286.85
4,Wired Headphones,20520,246034.80


In [17]:
top_sales = (
    product_summary
    .sort_values("total_sales", ascending=False)
    .head(5)
    .reset_index(drop=True)
)
top_sales[["product", "quantity_sold", "total_sales"]]

,product,quantity_sold,total_sales
0,Macbook Pro Laptop,4724,8030800.00
1,iPhone,6845,4791500.00
2,ThinkPad Laptop,4126,4125958.74
3,Google Phone,5528,3316800.00
4,27in 4K Gaming Monitor,6238,2432757.62


**Highest quantity sold.** The five products with the highest `quantity_sold` are all cheap accessories: AAA Batteries (30,981), AA Batteries (27,615), USB-C Charging Cable (23,927), Lightning Charging Cable (23,163) and Wired Headphones (20,520). Their `total_sales` are low, from about $93,000 for the AAA Batteries to $346,287 for the Lightning Charging Cable.

**Highest total sales.** The five products with the highest `total_sales` are expensive electronics: the Macbook Pro Laptop ($8.03M), iPhone ($4.79M), ThinkPad Laptop ($4.13M), Google Phone ($3.32M) and 27in 4K Gaming Monitor ($2.43M). Their `quantity_sold` is much lower, between 4,126 and 6,845.

**Comparison.** The two top-five lists have no products in common. The AAA Batteries have the highest `quantity_sold` but bring in only about $93,000, while the Macbook Pro Laptop sold 4,724 items and brought in over $8 million. So the products that sell the most items are not the ones that earn the most money. The difference comes from the price of each item: a laptop at $1,700 earns more from one purchase than a battery pack at about $3 earns from hundreds. For the business, this means high-volume accessories and high-price electronics play different roles, and the electronics matter far more for total sales.

**Suggestion:** The store should keep the high-`quantity_sold` accessories always in stock to draw people in, and display the high-`total_sales` products where those customers can see them, since some may buy them too. 